# Gold Layer — dim_restaurant (SCD 2) + fact_video_restaurant_mention (design 1.3 / 1.4)

Source: `prod.silver.stg_restaurant_mentions` (LLM extraction). Run **after** `Analytic/gold/dims` (needs `dim_video`).

**Dedup → `restaurant_id` (durable):** nếu Google Places tìm thấy quán (`stg_restaurant_geocode`, `match_status = matched`) thì `restaurant_id = 'place:' || place_id` (cùng place = cùng quán, kể cả khi tên trong video khác nhau). Nếu không: cùng `restaurant_name_normalized` **và** cùng quận (`district` normalize: bỏ dấu, `Q.3` / `quận 3` → `quan 3`) → cùng quán; mention thiếu quận dùng quận duy nhất của tên đó nếu tên chỉ xuất hiện ở 1 quận.

**Address / location:** matched restaurants take Google's name, formatted address, ward, city, latitude / longitude and place_id; the district stays the one stated in the video (Google returns post-2025 wards and no district).

**SCD 2:** address / ward / district / city / latitude / longitude / place_id are tracked (`tracked_checksum`). A change closes the current version (`valid_to = run time`, `is_current = false`) and inserts a new one; a restaurant that is no longer in the source is closed without a new version; `restaurant_key = xxhash64(restaurant_id, valid_from)`. `restaurant_name` is **type 1** (overwrites every version).

**fact_video_restaurant_mention:** factless bridge, grain video × restaurant. `restaurant_key` = the version current when the mention was first found (kept on later runs). `mention_weight = 1 / restaurants in the video` (sums to 1 per video). Loaded with one full-sync MERGE (atomic).


In [ ]:
%run ./_udfs


In [ ]:
# --- Config ---
# --- Layer schemas: job parameters bronze / silver / gold (catalog.schema), default prod.* ---
for layer in ("bronze", "silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
BRONZE, SILVER, GOLD = (dbutils.widgets.get(layer) for layer in ("bronze", "silver", "gold"))

DIM = f"{GOLD}.dim_restaurant"
FACT = f"{GOLD}.fact_video_restaurant_mention"
GEOCODE = f"{SILVER}.stg_restaurant_geocode"  # Google Places lookups (stg/restaurant_geocode)
if not spark.catalog.tableExists(GEOCODE):
    raise RuntimeError(f"{GEOCODE} does not exist: run stg/restaurant_geocode first")

run_ts = spark.sql("SELECT current_timestamp() AS ts").collect()[0]["ts"]
print(f"Silver source: {SILVER}.stg_restaurant_mentions")
print(f"Gold target: {DIM}, {FACT}")
print(f"🕒 run_ts (valid_from of new versions): {run_ts}")


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DIM} (
        restaurant_key             BIGINT,
        restaurant_id              STRING,
        restaurant_name            STRING,
        restaurant_name_normalized STRING,
        address                    STRING,
        ward                       STRING,
        district                   STRING,
        city                       STRING,
        latitude                   DOUBLE,
        longitude                  DOUBLE,
        place_id                   STRING,
        valid_from                 TIMESTAMP,
        valid_to                   TIMESTAMP,
        is_current                 BOOLEAN,
        tracked_checksum           STRING,
        etl_date                   TIMESTAMP
    )
    USING DELTA
    COMMENT 'Restaurant dimension (SCD 2 on address / location, name type 1). One row per version, -1 = Unknown.'
""")

# --- Every mention with its durable restaurant_id (dedup rule above) ---
spark.sql(rf"""
    CREATE OR REPLACE TEMP VIEW mentions_keyed AS
    WITH m AS (
        SELECT *,
            nullif(trim(regexp_replace(regexp_replace(regexp_replace(
                lower(unaccent_vi(coalesce(district, ''))),
                '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
                '^(tp\\.?|thanh pho)\\s+', ''),
                '\\s+', ' ')), '') AS district_norm
        FROM {SILVER}.stg_restaurant_mentions
    ),
    single_district AS (
        SELECT restaurant_name_normalized, max(district_norm) AS only_district
        FROM m WHERE district_norm IS NOT NULL
        GROUP BY restaurant_name_normalized HAVING COUNT(DISTINCT district_norm) = 1
    )
    SELECT m.*,
           g.place_id, g.place_name, g.formatted_address, g.ward AS geo_ward, g.city AS geo_city,
           g.latitude, g.longitude,
           -- same Google place = same restaurant; otherwise normalized name + district
           COALESCE('place:' || g.place_id,
                    md5(concat(m.restaurant_name_normalized, '|', coalesce(m.district_norm, s.only_district, '')))) AS restaurant_id
    FROM m
    LEFT JOIN single_district s ON s.restaurant_name_normalized = m.restaurant_name_normalized
    LEFT JOIN {GEOCODE} g
      ON g.restaurant_name_normalized = m.restaurant_name_normalized AND g.district_norm <=> m.district_norm
     AND g.match_status = 'matched'
""")

# --- One row per restaurant_id: latest mention wins (prefer rows with an address) ---
spark.sql("""
    CREATE OR REPLACE TEMP VIEW restaurant_src AS
    WITH ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY restaurant_id ORDER BY place_id IS NULL, address IS NULL, extracted_at DESC, video_id) AS rn
        FROM mentions_keyed
    )
    SELECT restaurant_id,
           COALESCE(place_name, restaurant_name)    AS restaurant_name,
           restaurant_name_normalized,
           COALESCE(formatted_address, address)     AS address,
           COALESCE(geo_ward, ward)                 AS ward,
           district,
           COALESCE(geo_city, city)                 AS city,
           latitude, longitude, place_id,
           md5(concat_ws('||', COALESCE(formatted_address, address), COALESCE(geo_ward, ward), district,
                         COALESCE(geo_city, city), CAST(latitude AS STRING), CAST(longitude AS STRING), place_id)) AS tracked_checksum
    FROM ranked WHERE rn = 1
""")
spark.sql("""
    CREATE OR REPLACE TEMP VIEW mention_restaurant AS
    SELECT DISTINCT video_id, restaurant_id, mention_source FROM mentions_keyed
""")
print(f"Restaurants in source: {spark.table('restaurant_src').count()}")


In [ ]:
# --- SCD 2 MERGE: close the changed current version and insert its new version in one atomic MERGE ---
spark.sql(f"""
    MERGE INTO {DIM} AS t
    USING (
        SELECT s.restaurant_id AS merge_key, s.* FROM restaurant_src s
        UNION ALL
        -- changed restaurants appear a second time with a NULL key, so they also hit WHEN NOT MATCHED (new version)
        SELECT NULL AS merge_key, s.*
        FROM restaurant_src s
        JOIN {DIM} d ON d.restaurant_id = s.restaurant_id AND d.is_current AND d.tracked_checksum <> s.tracked_checksum
    ) AS s
    ON t.restaurant_id = s.merge_key AND t.is_current
    WHEN MATCHED AND t.tracked_checksum <> s.tracked_checksum THEN
        UPDATE SET t.valid_to = TIMESTAMP'{run_ts}', t.is_current = false, t.etl_date = current_timestamp()
    WHEN NOT MATCHED THEN INSERT (
        restaurant_key, restaurant_id, restaurant_name, restaurant_name_normalized, address, ward, district, city,
        latitude, longitude, place_id, valid_from, valid_to, is_current, tracked_checksum, etl_date
    ) VALUES (
        xxhash64(s.restaurant_id, TIMESTAMP'{run_ts}'), s.restaurant_id, s.restaurant_name, s.restaurant_name_normalized,
        s.address, s.ward, s.district, s.city, s.latitude, s.longitude, s.place_id,
        TIMESTAMP'{run_ts}', TIMESTAMP'9999-12-31 00:00:00', true, s.tracked_checksum, current_timestamp()
    )
    -- a restaurant that left the source (e.g. its mentions now resolve to a Google place_id): close it
    WHEN NOT MATCHED BY SOURCE AND t.is_current AND t.restaurant_key <> -1 THEN
        UPDATE SET t.valid_to = TIMESTAMP'{run_ts}', t.is_current = false, t.etl_date = current_timestamp()
""")

# --- Type 1: the display name overwrites every version ---
spark.sql(f"""
    MERGE INTO {DIM} AS t
    USING restaurant_src AS s ON t.restaurant_id = s.restaurant_id
    WHEN MATCHED AND NOT (t.restaurant_name <=> s.restaurant_name) THEN
        UPDATE SET t.restaurant_name = s.restaurant_name, t.etl_date = current_timestamp()
""")
spark.sql(f"""
    INSERT INTO {DIM} (restaurant_key, restaurant_name, valid_from, valid_to, is_current, etl_date)
    SELECT -1, 'Unknown', TIMESTAMP'1900-01-01 00:00:00', TIMESTAMP'9999-12-31 00:00:00', true, current_timestamp()
    WHERE NOT EXISTS (SELECT 1 FROM {DIM} WHERE restaurant_key = -1)
""")
v = spark.sql(f"SELECT COUNT(*) AS n, COUNT_IF(is_current) AS cur FROM {DIM} WHERE restaurant_key <> -1").collect()[0]
print(f"✅ {DIM}: {v['n']} versions, {v['cur']} current")


In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {FACT} (
        video_key      BIGINT,
        restaurant_key BIGINT,
        mention_source STRING,
        mention_count  TINYINT,
        mention_weight DOUBLE,
        etl_date       TIMESTAMP
    )
    USING DELTA
    COMMENT 'Factless bridge: 1 video x 1 restaurant. mention_weight = 1 / restaurants in the video (sums to 1).'
""")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW fact_mention_src AS
    WITH m AS (
        SELECT video_id, restaurant_id, max(mention_source) AS mention_source
        FROM mention_restaurant GROUP BY video_id, restaurant_id
    ),
    existing AS (   -- keep the version that was current when the mention was first found
        SELECT f.video_key, f.restaurant_key, d.restaurant_id
        FROM {FACT} f JOIN {DIM} d ON d.restaurant_key = f.restaurant_key
    )
    SELECT
        COALESCE(dv.video_key, -1)                              AS video_key,
        COALESCE(e.restaurant_key, cur.restaurant_key, -1)      AS restaurant_key,
        m.mention_source,
        CAST(1 AS TINYINT)                                      AS mention_count,
        1.0D / COUNT(*) OVER (PARTITION BY m.video_id)          AS mention_weight,
        current_timestamp()                                     AS etl_date
    FROM m
    LEFT JOIN {GOLD}.dim_video dv ON dv.video_id = m.video_id
    LEFT JOIN existing e ON e.video_key = dv.video_key AND e.restaurant_id = m.restaurant_id
    LEFT JOIN {DIM} cur ON cur.restaurant_id = m.restaurant_id AND cur.is_current
""")
spark.sql(f"""
    MERGE INTO {FACT} AS t
    USING fact_mention_src AS s ON t.video_key = s.video_key AND t.restaurant_key = s.restaurant_key
    WHEN MATCHED AND NOT (t.mention_weight <=> s.mention_weight AND t.mention_source <=> s.mention_source) THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
    WHEN NOT MATCHED BY SOURCE THEN DELETE
""")
f = spark.sql(f"SELECT COUNT(*) AS n, COUNT(DISTINCT video_key) AS videos, COUNT(DISTINCT restaurant_key) AS restaurants FROM {FACT}").collect()[0]
print(f"✅ {FACT}: {f['n']} mentions, {f['videos']} videos, {f['restaurants']} restaurants")
print("\n--- dim_restaurant + fact_video_restaurant_mention complete ---")
